In [0]:
import re

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.dropdown(
    "schema",
    "migration_dev",
    ["migration_dev", "migration_test", "migration_prod_sim"],
)
dbutils.widgets.text("batch_id", "batch_001")

catalog_name = dbutils.widgets.get("catalog").strip()
schema_name = dbutils.widgets.get("schema").strip()
batch_id = dbutils.widgets.get("batch_id").strip()

if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog_name):
    raise ValueError("Invalid catalog name")

if schema_name not in {
    "migration_dev", "migration_test", "migration_prod_sim"
}:
    raise ValueError("Invalid environment schema")

if not re.fullmatch(r"batch_[0-9]+", batch_id):
    raise ValueError("Invalid batch ID")

table_prefix = f"{catalog_name}.{schema_name}"
volume_root = f"/Volumes/{catalog_name}/{schema_name}"

print(f"Environment: {table_prefix}")
print(f"Batch: {batch_id}")
print(f"Volume root: {volume_root}")

In [0]:
import json
import hashlib
from pathlib import Path
from pyspark.sql import functions as F


batch_path = Path(f"{volume_root}/landing/{batch_id}")

csv_path = batch_path / "metadata" / "statements.csv"
manifest_path = batch_path / "manifest.json"

# Only ingest a batch declared complete by the source.
if not manifest_path.is_file():
    raise RuntimeError("Batch is not ready: manifest is missing")

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

if manifest["status"] != "READY" or manifest["batch_id"] != batch_id:
    raise RuntimeError("Manifest status or batch ID is invalid")

actual_csv_hash = hashlib.sha256(csv_path.read_bytes()).hexdigest()

if actual_csv_hash != manifest["metadata_sha256"]:
    raise RuntimeError("CSV has changed since the manifest was created")

# Preserve source values as strings in Bronze.
source_schema = """
    batch_id STRING,
    document_id STRING,
    account_number STRING,
    customer_name STRING,
    document_type STRING,
    statement_date STRING,
    file_name STRING,
    pdf_path STRING,
    source_sha256 STRING,
    _rescued_data STRING
"""

state_path = f"{volume_root}/checkpoints/bronze_metadata/{batch_id}"

raw = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("pathGlobFilter", "*.csv")
    .option("cloudFiles.schemaLocation", f"{state_path}/schema")
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("rescuedDataColumn", "_rescued_data")
    .schema(source_schema)
    .load(str(batch_path / "metadata"))
)

bronze = raw.select(
    "*",
    F.col("_metadata.file_path").alias("source_file"),
    F.col("_metadata.file_modification_time")
        .alias("source_modification_time"),
    F.current_timestamp().alias("ingestion_timestamp"),
)

query = (
    bronze.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{state_path}/stream")
    .trigger(availableNow=True)
    .toTable(f"{table_prefix}.bronze_metadata")
)

query.awaitTermination()

saved = (
    spark.table(f"{table_prefix}.bronze_metadata")
    .filter(F.col("batch_id") == batch_id)
)

print(f"Bronze records for {batch_id}: {saved.count()}")
display(saved.orderBy("document_id").limit(5))

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT document_id) AS unique_documents,
    COUNT(DISTINCT source_file) AS source_csv_files,
    COUNT_IF(_rescued_data IS NOT NULL) AS rescued_rows
FROM IDENTIFIER(:catalog || '.' || :schema || '.bronze_metadata')
WHERE batch_id = :batch_id;